# Token surprisal for typo detection

GPT-2 assigns probabilities to the next token given its context. I use negative log probability as an anomaly score, comparing the logits at position t−1 to token t.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from nlp_portfolio.generation import token_surprisals, calibrate_threshold

model_id = 'gpt2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
print('Device:', device)

In [ ]:
sentences = [
    'The researcher published the results yesterday.',
    'The researhcer published the results yesterday.',
]
for text in sentences:
    token_scores = token_surprisals(text, tokenizer, model)
    print(text)
    print([(token, round(score, 2)) for token, score in token_scores])

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

example_sentences = [
    'The scientist opened the notebook.', 'My friend reads books.',
    'The weather is pleasant today.', 'This is a normal sentence.',
    'The scientst opened the notebook.', 'My freind reads books.',
    'The wheather is pleasant today.', 'Ths is a normal sentence.',
]
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])
scores = np.array([max((v for _, v in token_surprisals(t, tokenizer, model)), default=0) for t in example_sentences])
cal_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.5, random_state=42, stratify=y)
threshold = calibrate_threshold(scores[cal_idx], y[cal_idx])
print('Calibrated threshold:', threshold)
print(classification_report(y[test_idx], scores[test_idx] >= threshold, zero_division=0))

The small examples check the scoring implementation and calibration split. Rare technical words can have high surprisal without being misspelled, so this approach needs a larger labelled test set before reporting general performance.